# Create Wolf Prize Awards (PRIZE PATTERN)

Wolf Prize laureates from wolffund.org.il via the WordPress REST API
(category 27 = "the-wolf-prize-winners"). 391 winners spanning fields
Physics, Chemistry, Mathematics, Medicine, Agriculture, Architecture,
Music, Painting & Sculpture, Leadership.

**Awarding body:** Wolf Foundation (Israel) — F4320320951

**Prerequisites:**
- Run `scripts/local/wolf_to_s3.py` first.

**S3:** `s3a://openalex-ingest/awards/wolf_prize/wolf_prize_winners.parquet`

Schema notes (PRIZE PATTERN):
- One row per laureate. Field = funder_scheme. Year from post date.
- amount: $100K USD per Wolf Prize (constant historical) — we *could*
  hardcode it but Wolf Foundation has occasionally shifted the value.
  We leave amount NULL and document with a notebook header note;
  the Step 6.7 check is waived as for HHMI.
- currency = "USD".
- Affiliation parsed from post content ("Affiliation at the time of
  the award:"). Citation captured separately.


## Step 1: Create staging table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wolf_prize_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/wolf_prize/wolf_prize_winners.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.wolf_prize_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.wolf_prize_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.wolf_prize_raw LIMIT 5;

## Step 2: Transform

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wolf_prize_awards
USING delta
AS
WITH wolf_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320951
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':wolf:', LOWER(s.slug)))) % 9000000000 AS id,
    CONCAT('Wolf Prize in ', COALESCE(s.field, 'Various'), ' ', CAST(s.year AS STRING),
           ' — ', s.name) AS display_name,
    NULLIF(s.citation, '') AS description,
    f.funder_id,
    s.slug AS funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,         -- per-laureate share not published
    'USD' AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name, f.ror_id, f.doi
    ) AS funder,
    'prize' AS funding_type,
    s.field AS funder_scheme,
    'wolf_prize_wp' AS provenance,
    TRY_TO_DATE(CONCAT(CAST(s.year AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(CAST(s.year AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
    s.year AS start_year,
    s.year AS end_year,
    -- Use the given_name/family_name columns the script now provides directly
    -- (last whitespace-token = family). Cleaner than the previous SQL-side regex
    -- which surfaced "P. Eisenstein" as the family for "James P. Eisenstein".
    struct(
        s.given_name AS given_name,
        s.family_name AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            NULLIF(s.affiliation, '') AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    s.url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':wolf:', LOWER(s.slug)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.wolf_prize_raw s
CROSS JOIN wolf_funder f
WHERE s.slug IS NOT NULL AND s.year IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 47

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'wolf_prize_wp' AND priority = 47;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id,
       amount, currency, funder, funding_type, funder_scheme, provenance,
       start_date, end_date, start_year, end_year,
       lead_investigator, co_lead_investigator, investigators,
       landing_page_url, doi, works_api_url,
       created_date, updated_date,
       47 AS priority
FROM openalex.awards.wolf_prize_awards;

## Verification

In [ ]:
%sql
-- Step 6.7 amount/currency check — WAIVED for Wolf (per-laureate amount not published).
-- Expect pct_amount = 0%, distinct_currencies = 1.
SELECT COUNT(*) total, COUNT(amount) has_amount,
       ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) pct_amount,
       collect_set(currency) currencies,
       COUNT(DISTINCT funder_scheme) distinct_fields
FROM openalex.awards.wolf_prize_awards;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 80) AS title, funder_scheme AS field,
       start_year, lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name AS institution
FROM openalex.awards.wolf_prize_awards
ORDER BY start_year DESC LIMIT 10;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) cnt
FROM openalex.awards.wolf_prize_awards GROUP BY funder_scheme ORDER BY cnt DESC;